# పాఠం 05 - ఏజెంటిక్ RAG


## సెటప్

ఈ నోట్‌బుక్ Microsoft Agent Framework ఉపయోగించి Agentic RAG (Retrieval-Augmented Generation) నమూనాను చూపిస్తుంది.

**ముందస్తు అవసరాలు:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — మీ Azure AI Search సేవా ఎండ్పాయింట్
- `AZURE_SEARCH_API_KEY` — మీ Azure AI Search API కీ
- పర్యావరణ చరాలు ద్వారా కాన్ఫిగర్ చేసిన Azure OpenAI డిప్లాయ్‌మెంట్
- Azure CLI గుర్తింపు (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ఏజెంటిక్ RAG అంటే ఏమిటి?

సాంప్రదాయ RAG ఒక స్థిరమైన పైప్‌లైన్‌ను అనుసరిస్తుంది: డాక్యుమెంట్లను పొందండి, ఆపై స్పందనను రూపొందించండి. **ఏజెంటిక్ RAG** సమాచారం తెప్పించేందుకు ఏజెంట్‌కు **ఎప్పుడు** మరియు **ఎలా** నిర్ణయించుకునే స్వేచ్ఛను ఇస్తూ ముందుకు వెళ్తుంది.

ఏజెంటిక్ RAGతో, ఏజెంట్ చేయగలదు:
- ప్రశ్నకు సమాధానం ఇవ్వడానికి ముందు తెప్పింపు అవసరమా అని **నిర్ణయించడం**
- ఏ డేటా మూలం లేదా సాధనాన్ని ప్రశ్నించాలో **ఎంచుకోవడం**
- తెప్పించిన ఫలితాలను **అంచనా వేయడం** మరియు మొదటి ప్రయత్నం తక్కువగా ఉంటే అనుసరణ తెప్పింపులను చేయడం
- అనేక తెప్పింపు దశల నుండి సమాచారాన్ని కలిపి సమగ్ర సమాధానాన్ని **సృష్టించడం**

ఇది ఏజెంట్‌ను ఒక స్థిరమైన తెప్పించి-తర్వాత-సృష్టించే పైప్‌లైన్ కంటే ఎక్కువ సడలింపు మరియు ఖచ్చితత్వం కలిగినది చేస్తుంది.


## శోధన సాధనం సృష్టించడం

Agentic RAGలో, బాహ్య డేటా వనరులను ఏజెంట్ అభ్యర్థిస్తే పిలువగల **సాధనాలు** గా ముట్టడించబడతాయి. ఇది ఏజెంట్ కు పొందికను ఒక తప్పనిసరి దశగా కాకుండా, అది తీసుకోవచ్చని మరో చర్యగా పొందుపరుస్తుంది.

కింద ఒక ప్రయాణ జ్ఞాన బేస్ ను నిర్వచించి, దానిని ఏజెంట్ గమ్యస్థాన సమాచారాన్ని చూసుకునేందుకు పిలవగల సాధనంగా ప్రదర్శిస్తాము.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ఏజెంట్‌ను నిర్మించడం

ఇప్పుడు మేము ఒక ఏజెంట్‌ను సృష్టిస్తాము, ఇది **ప్రతిస్పందన ఇచ్చేముందు ఎల్లప్పుడూ సమాచారం తీసుకోవాలని సూచించబడింది**. ఏజెంట్ తన స్వంత శిక్షణ డేటాకు ఆధారపడకపోవడం కంటే, జ్ఞానాధారంలో తన ప్రతిస్పందనలను స్థాపించే కోసం `search_travel_knowledge` టూల్ ను ఉపయోగిస్తుంది.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — మేకర్-చెక్కర్ ప్యాటర్న్

Agentic RAG యొక్క ఒక ప్రధాన లాభం **పునరావృత రిట్రీవల్**. ఏజెంట్ తన ప్రాథమిక అంశాలను ధృవీకరించడం, మెరుగు చేయడం లేదా విస్తరించడం కోసం అనేక రౌండ్లు శోధన చేయవచ్చు — ఇది "మేకర్-చెక్కర్" వర్క్‌ఫ్లోకు సమానమే:

1. **మేకర్ స్టెప్**: ఏజెంట్ ప్రారంభ సమాచారం తీసుకొని ప్రతిస్పందనను ముసాయిదా చేస్తుంది.
2. **చెక్కర్ స్టెప్**: ఏజెంట్ వివరాలను ధృవీకరించడానికి లేదా గ్యాప్‌లను పూరించడానికి అదనపు రిట్రీవల్స్ చేస్తుంది.

క్రింద, ఏజెంట్‌ను అనేక గమ్యస్థానాలను పోల్చే ప్రశ్న అడగబడింది, ఇది అనేక సార్లు శోధించమని ప్రేరేపిస్తుంది.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ఉపయోగించి **Agentic RAG** సిస్టమ్‌ను ఎలా నిర్మించాలో నేర్చుకున్నారు:

- **Agentic RAG** ఏజెంట్లు సమాచారాన్ని ఎప్పుడు తిరిగి పొందాలో స్వయం निर्णయం తీసుకునేలా చేస్తుంది, ఇది తిరిగి పొందడం స్థిరంగా కాకుండా ప్రేరేపకంగా చేస్తుంది.
- **పరికరాలు డేటా మూలాలుగా**: Azure AI Search వంటి బాహ్య జ్ఞాన బేస్‌లు ఏజెంట్ పిలవగల పరికరాలుగా నింపబడ్డాయి.
- **పునరావృత తిరిగి పొందడం**: తయారీదారు-పరిశీలకుడి నమూనా ఏజెంట్‌కు మరొకటి పై మరొకటి తిరిగి పొందే దశలను నిర్వహించడానికి వీలు కల్పిస్తుంది — శోధించడం, నిర్ధారించడం, మరియు మెరుగుపరచడం — తుది సమాధానం ఇవ్వడానికి ముందు.

ఉత్పత్తిలో, మీరు భారీ స్థాయి ప్రయాణా పత్రాల తిరిగి పొందడానికి మేమరీలో ఉన్న `TRAVEL_KNOWLEDGE_BASE`ను వాస్తవ Azure AI Search సూచికతో మార్చుతారు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**తప్పింపు**:  
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించడం జరిగింది. సరిగ్గా అనువదించడానికి ప్రయత్నిస్తున్నప్పటికీ, ఆబోమేటెడ్ అనువాదాల్లో తప్పులు లేదా లోపాలు ఉండవచ్చు. ఒరిజినల్ పత్రం మాతృభాషలో ఉన్నది అధికారిక మూలం కాబట్టి దాన్ని ఆధారంగా తీసుకోవాలి. కీలకమైన సమాచారానికి, వృత్తిపరమైన మానవ అనువాదం చేయించుకోవడం మంచిది. ఈ అనువాదం వలన ఏవైనా తప్పుదర్యాప్తులు లేదా పొరపాట్లకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
